# FITILA — entraînement du traducteur FR ↔ Baatɔnum (ByT5)
Kaggle : *Accelerator* = GPU T4 x2 ou P100, *Internet* = On. Ajoutez le dataset `dunya-translate-kit` (le zip fourni).
Optionnel : secret Kaggle `HF_TOKEN` pour repartir des poids v03 (sinon `google/byt5-small`).

In [ ]:
import os, glob, subprocess, sys
kit = glob.glob('/kaggle/input/*/dunya-translate-kit*')[:1] or glob.glob('/kaggle/input/*')[:1]
print(kit)
!pip -q install transformers sentencepiece accelerate sacrebleu
!mkdir -p /kaggle/working/kit && cp -r /kaggle/input/*/* /kaggle/working/kit/ 2>/dev/null; ls /kaggle/working/kit

In [ ]:
import os
ROOT = next(p for p in ['/kaggle/working/kit', '/kaggle/working/kit/dunya-translate-kit'] if os.path.exists(p + '/scripts'))
os.chdir(ROOT + '/scripts/dunya_translate')
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
MODEL = 'zimesongbian/modele_byt5_bariba_expert_api_v03_improve' if os.environ.get('HF_TOKEN') else 'google/byt5-small'
LR = '1e-4' if os.environ.get('HF_TOKEN') else '3e-4'
print('base model:', MODEL)

In [ ]:
!python -m unittest test_pipeline 2>&1 | tail -3
!python make_training_data.py

## 1. Fine-tuning supervisé (indices mémoire + glossaire, Apprendre multitâche)

In [ ]:
!python train_byt5.py --model $MODEL --out /kaggle/working/v04 --epochs 6 --lr $LR --batch 8 --accum 4

## 2. Renforcement : exemples difficiles puis préférences (DPO)

In [ ]:
!python mine_hard_examples.py --model /kaggle/working/v04 --samples 6 --limit 4000
!python dpo.py --model /kaggle/working/v04 --out /kaggle/working/v04-dpo --epochs 1 --batch 8

## 3. Mesure sur le jeu de test (mémoire = train seulement)

In [ ]:
!python eval_translation.py --backend hf --model /kaggle/working/v04 --rag --out /kaggle/working/eval_v04_rag.json | head -40
!python eval_translation.py --backend hf --model /kaggle/working/v04-dpo --rag --out /kaggle/working/eval_v04_dpo_rag.json | head -40

## 4. Export ONNX hors ligne + archive à télécharger
Comparer chrF++ avec la ligne de base v03 (26,0 FR→BA / 23,6 BA→FR avec autocorrection ; 26,6 / 26,9 brut).

In [ ]:
!pip -q install onnx onnxruntime
!python ../dunya_byt5/export_byt5_onnx.py --model /kaggle/working/v04-dpo --out /kaggle/working/onnx --sample 'fr-ba: Bonjour' --sample 'ba-fr: A kua'
!cd /kaggle/working && zip -qr dunya-translate-result.zip v04-dpo onnx eval_*.json && ls -la dunya-translate-result.zip